# Day 3. Generative LLMs for wrangling and crawling

**Instructor notebook.** Runs on paid Colab Pro. The crawl uses generic Python `requests` and does not need a GPU. The Anthropic API calls happen over the network.

Today we change tools. Days 1 and 2 used encoder models for closed-label classification. Today we use a generative model, Anthropic Claude Haiku 4.5, for three jobs encoders cannot easily do. Free-text extraction, normalization, and lightweight summarization. Then we crawl a public site politely and feed the results back through the extractor for an end-to-end pipeline.

**Honest framing up front.** This notebook builds a structured wrangling pipeline, not a measurement. We do not build a hand-coded gold set today. Section 6 explains what you can validate without one. Slide 28 of the deck reminds you that a paper still needs the gold set. Today's notebook is the template you reuse when you have the budget for one.

**Before you run.** Make sure `news_leads.csv` is uploaded to your Google Drive at `MyDrive/day3/news_leads.csv`, and that `ANTHROPIC_API_KEY` is saved under the key icon in the Colab sidebar.


## 1. Setup

Mount Drive, install libraries, pull the API key from `userdata`. Never paste the key into a code cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q anthropic requests beautifulsoup4


In [ ]:
import os
import json
import time
import hashlib
import random
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from anthropic import Anthropic
client = Anthropic()
print('Anthropic client ready.')


## 2. Load `news_leads.csv`

Fifty short news leads about international agreements. Two columns. `lead_id` and `text`. The leads are constructed examples written for instructional use, varied across sectors and types, with some clean and some intentionally messy.


In [ ]:
CSV_PATH = '/content/drive/MyDrive/day3/news_leads.csv'
leads = pd.read_csv(CSV_PATH)
print(f'Shape: {leads.shape}')
print()
for _, row in leads.head(3).iterrows():
    print(f"[{row['lead_id']}] {row['text']}")
    print()


## 3. Define the schema

Four fields. `signatories` is a list of strings. `agreement_type` is one of a closed taxonomy. `year` is a four-digit integer or `null`. `sector` is one of a closed taxonomy.

The taxonomies are pinned in the system prompt. Pinning them keeps the model from inventing reasonable but inconsistent values across calls.


In [ ]:
AGREEMENT_TYPES = [
    'bilateral_trade', 'multilateral_trade', 'climate', 'defense',
    'health', 'financial_assistance', 'IP', 'security_cooperation',
    'infrastructure', 'technology', 'diplomatic_normalization', 'other',
]

SECTORS = [
    'trade', 'climate', 'defense', 'health', 'finance', 'IP', 'security',
    'infrastructure', 'technology', 'energy', 'agriculture', 'migration', 'other',
]

SYSTEM_PROMPT = f\"\"\"You are an information extraction system. The user will provide a single short news lead about an international agreement. Extract a JSON object with exactly four fields.

  "signatories": list of strings. The parties to the agreement, as named in the text. Use the names exactly as they appear, but normalize country references to standard short names (for example, "USA", "U.S." -> "United States").
  "agreement_type": one of {AGREEMENT_TYPES}.
  "year": a four-digit integer for the year the agreement was signed or announced, or null if the text does not specify.
  "sector": one of {SECTORS}.

Return ONLY the JSON object. Do not wrap it in markdown. Do not add commentary.

Example input:
"On 12 March 2024, France and Kenya signed a bilateral health-sector aid agreement worth EUR 80 million."

Example output:
{{"signatories": ["France", "Kenya"], "agreement_type": "health", "year": 2024, "sector": "health"}}
\"\"\"

print(SYSTEM_PROMPT[:500])
print('...')
print(f'\\nSystem prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'Agreement types: {len(AGREEMENT_TYPES)}')
print(f'Sectors: {len(SECTORS)}')


## 4. Single-record extraction

The Anthropic call. System prompt holds the instructions. User prompt is the lead text. Temperature is zero for reproducibility.


In [ ]:
MODEL_ID = 'claude-haiku-4-5-20251001'

def extract_one(lead_text, max_tokens=300):
    \"\"\"Call Claude on one lead. Return parsed JSON dict and the raw response.\"\"\"
    resp = client.messages.create(
        model=MODEL_ID,
        max_tokens=max_tokens,
        temperature=0,
        system=[
            {'type': 'text', 'text': SYSTEM_PROMPT, 'cache_control': {'type': 'ephemeral'}},
        ],
        messages=[{'role': 'user', 'content': lead_text}],
    )
    text = resp.content[0].text
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        data = None
    return data, resp

# Run on the first lead and inspect
sample = leads.iloc[0]
data, resp = extract_one(sample['text'])
print(f"Lead [{sample['lead_id']}]:")
print(f"  {sample['text']}\\n")
print('Extracted JSON:')
print(json.dumps(data, indent=2))
print(f'\\nInput tokens:  {resp.usage.input_tokens}')
print(f'Output tokens: {resp.usage.output_tokens}')
if hasattr(resp.usage, 'cache_creation_input_tokens'):
    print(f'Cache creation tokens: {resp.usage.cache_creation_input_tokens}')
if hasattr(resp.usage, 'cache_read_input_tokens'):
    print(f'Cache read tokens:    {resp.usage.cache_read_input_tokens}')


## 5. Schema validators

Programmatic checks. No hand-coding required. The validators answer a different question from gold-set F1, and they answer it for free.


In [ ]:
def validate_record(rec):
    \"\"\"Return (is_valid, list of error strings).\"\"\"
    errors = []
    if not isinstance(rec, dict):
        return False, ['not a dict']
    required = {'signatories', 'agreement_type', 'year', 'sector'}
    missing = required - set(rec.keys())
    if missing:
        errors.append(f'missing fields: {sorted(missing)}')
    extra = set(rec.keys()) - required
    if extra:
        errors.append(f'extra fields: {sorted(extra)}')
    sig = rec.get('signatories')
    if not isinstance(sig, list) or not all(isinstance(s, str) for s in sig or []):
        errors.append('signatories must be list of strings')
    elif len(sig) == 0:
        errors.append('signatories is empty')
    at = rec.get('agreement_type')
    if at not in AGREEMENT_TYPES:
        errors.append(f'agreement_type {at!r} not in taxonomy')
    yr = rec.get('year')
    if yr is not None and not (isinstance(yr, int) and 1990 <= yr <= 2030):
        errors.append(f'year {yr!r} not int in [1990, 2030] or null')
    sec = rec.get('sector')
    if sec not in SECTORS:
        errors.append(f'sector {sec!r} not in taxonomy')
    return len(errors) == 0, errors

ok, errs = validate_record(data)
print(f'Validators on L001: passed = {ok}')
if errs:
    for e in errs:
        print(f'  - {e}')


## 6. Run on all 50 with audit logging

Every call writes one row to the audit log. The lead, the model ID, the system-prompt hash, the timestamp, the response, the input and output token counts, the validators that fired. This is what the replication package ships.


In [ ]:
SYSTEM_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:16]

def extract_with_log(lead_text, lead_id):
    t0 = time.time()
    data, resp = extract_one(lead_text)
    ok, errs = validate_record(data) if data is not None else (False, ['JSON parse failed'])
    cache_read = getattr(resp.usage, 'cache_read_input_tokens', 0) or 0
    cache_create = getattr(resp.usage, 'cache_creation_input_tokens', 0) or 0
    return {
        'lead_id': lead_id,
        'lead_text': lead_text,
        'model_id': MODEL_ID,
        'system_prompt_hash': SYSTEM_HASH,
        'temperature': 0,
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'wall_time_sec': round(time.time() - t0, 2),
        'response_text': resp.content[0].text,
        'parsed_json': json.dumps(data) if data is not None else None,
        'input_tokens': resp.usage.input_tokens,
        'output_tokens': resp.usage.output_tokens,
        'cache_read_tokens': cache_read,
        'cache_creation_tokens': cache_create,
        'validators_passed': ok,
        'validator_errors': '; '.join(errs) if errs else '',
    }

print(f'Running extraction on {len(leads)} leads...')
t0 = time.time()
log_rows = [extract_with_log(r.text, r.lead_id) for _, r in leads.iterrows()]
print(f'Done in {time.time()-t0:.1f} sec')

audit = pd.DataFrame(log_rows)
print(f'\\nSchema-validation pass rate: {audit.validators_passed.mean()*100:.1f}% ({audit.validators_passed.sum()} of {len(audit)})')
print(f'Total input tokens:  {audit.input_tokens.sum():>7}')
print(f'Cache read tokens:   {audit.cache_read_tokens.sum():>7}  (billed at ~10% of base rate)')
print(f'Total output tokens: {audit.output_tokens.sum():>7}')

if (~audit.validators_passed).any():
    print('\\nFirst 3 validator failures:')
    for _, r in audit[~audit.validators_passed].head(3).iterrows():
        print(f"  [{r.lead_id}] {r.validator_errors}")
        print(f"    response: {r.response_text[:160]}...")


## 7. Determinism check

Run the same ten leads twice at temperature zero. Count how many produce identical JSON. The drift you see is the floor on reproducibility for this model and prompt.


In [ ]:
subset = leads.sample(10, random_state=42).reset_index(drop=True)

def extract_text_only(lead_text):
    data, _ = extract_one(lead_text)
    return json.dumps(data, sort_keys=True) if data is not None else None

print('Running pass A...')
pass_a = [extract_text_only(r.text) for _, r in subset.iterrows()]
print('Running pass B...')
pass_b = [extract_text_only(r.text) for _, r in subset.iterrows()]

identical = sum(1 for a, b in zip(pass_a, pass_b) if a == b and a is not None)
print(f'\\nDeterminism: {identical} of {len(subset)} identical across two runs at temperature 0.')
if identical < len(subset):
    print('\\nDifferences:')
    for r, a, b in zip(subset.itertuples(), pass_a, pass_b):
        if a != b:
            print(f'  [{r.lead_id}]')
            print(f'    A: {a}')
            print(f'    B: {b}')


## 8. Sample audit

Read five random outputs by hand. The smallest hand-coding budget that still tells you whether the pipeline is sane. About ten minutes of human time.


In [ ]:
sample_idx = audit.sample(5, random_state=7).index
for i in sample_idx:
    r = audit.loc[i]
    print(f'\\n[{r.lead_id}]  validators_passed={r.validators_passed}')
    print(f'  text: {r.lead_text}')
    print(f'  json: {r.parsed_json}')


## 9. Robots.txt and the polite-fetch helper

The two-screen check that decides whether you can crawl a site. The five-line Python that does the actual work.


In [ ]:
import requests
from bs4 import BeautifulSoup

# Step 1. Fetch and read robots.txt on screen.
robots = requests.get('https://www.csis.org/robots.txt', timeout=10)
print(f'Status: {robots.status_code}')
print(robots.text[:1200])


In [ ]:
USER_AGENT = 'workshop-demo (mailto:jihye.park.psci@gmail.com)'

def polite_fetch(url, timeout=10):
    \"\"\"GET with a real UA, status-code check, return raw HTML and metadata.\"\"\"
    t0 = time.time()
    try:
        r = requests.get(url, headers={'User-Agent': USER_AGENT}, timeout=timeout)
    except requests.RequestException as e:
        return {'url': url, 'status': None, 'error': str(e), 'html': None,
                'fetched_at': datetime.now(timezone.utc).isoformat(),
                'wall_time_sec': round(time.time() - t0, 2)}
    return {
        'url': url,
        'status': r.status_code,
        'html': r.text if r.status_code == 200 else None,
        'error': None if r.status_code == 200 else f'HTTP {r.status_code}',
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'wall_time_sec': round(time.time() - t0, 2),
        'bytes': len(r.content),
    }


## 10. Crawl a single CSIS analysis article

Fetch one. Parse the headline, date, and first three paragraphs. Inspect.


In [ ]:
SAMPLE_URL = 'https://www.csis.org/analysis/ai-and-future-mediation'

f = polite_fetch(SAMPLE_URL)
print(f"URL:    {f['url']}")
print(f"Status: {f['status']}")
print(f"Bytes:  {f.get('bytes', 0):,}")
print(f"Took:   {f['wall_time_sec']}s")

if f['html']:
    soup = BeautifulSoup(f['html'], 'html.parser')
    h1 = soup.find('h1')
    headline = h1.get_text(strip=True) if h1 else '(no h1 found)'
    paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True)]
    print(f"\\nHeadline: {headline}")
    print(f"Paragraphs found: {len(paragraphs)}")
    print(f"\\nFirst paragraph:\\n  {paragraphs[0][:300] if paragraphs else '(none)'}")


## 11. Crawl five articles, with rate limiting and raw-HTML storage

Five URLs from the CSIS analysis index. One-second sleep between fetches. Raw HTML written to disk so the parser is rerunnable without a re-crawl.

If any of these URLs has changed since the workshop was prepared, switch to the cached fallback HTML in the workshop GitHub repo under `day3/cache/`.


In [ ]:
ARTICLE_URLS = [
    'https://www.csis.org/analysis/ai-and-future-mediation',
    'https://www.csis.org/analysis/european-defense-investment-after-ukraine',
    'https://www.csis.org/analysis/critical-minerals-supply-chains-2024',
    'https://www.csis.org/analysis/asean-and-south-china-sea',
    'https://www.csis.org/analysis/global-health-financing-after-pandemic',
]

RAW_DIR = Path('/content/drive/MyDrive/day3/raw_html')
RAW_DIR.mkdir(parents=True, exist_ok=True)

manifest_rows = []
for url in ARTICLE_URLS:
    f = polite_fetch(url)
    slug = url.rstrip('/').split('/')[-1]
    raw_path = RAW_DIR / f"{slug}.html"
    if f['html']:
        raw_path.write_text(f['html'])
    manifest_rows.append({
        'url': url,
        'slug': slug,
        'status': f['status'],
        'fetched_at': f['fetched_at'],
        'wall_time_sec': f['wall_time_sec'],
        'bytes': f.get('bytes'),
        'raw_path': str(raw_path) if f['html'] else None,
        'error': f['error'],
    })
    time.sleep(1.0)  # politeness

manifest = pd.DataFrame(manifest_rows)
print(manifest[['slug', 'status', 'bytes', 'wall_time_sec']].to_string(index=False))


## 12. Parse the raw HTML

BeautifulSoup pulls headline, date, and the first three paragraphs from each saved file. Output is a parsed-text CSV with one row per article.


In [ ]:
def parse_article(raw_html):
    soup = BeautifulSoup(raw_html, 'html.parser')
    h1 = soup.find('h1')
    headline = h1.get_text(strip=True) if h1 else None
    date_str = None
    for s in soup.find_all(string=True):
        s = s.strip()
        if s and ', 20' in s and len(s) < 40 and any(m in s for m in
            ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']):
            date_str = s
            break
    paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 80]
    lead_text = ' '.join(paragraphs[:3])[:1200]
    return {'headline': headline, 'date_text': date_str, 'lead_text': lead_text,
            'n_paragraphs': len(paragraphs)}

parsed_rows = []
for _, m in manifest.iterrows():
    if m['raw_path'] is None:
        parsed_rows.append({'slug': m['slug'], 'headline': None, 'date_text': None,
                            'lead_text': None, 'n_paragraphs': 0})
        continue
    raw = Path(m['raw_path']).read_text()
    parsed = parse_article(raw)
    parsed['slug'] = m['slug']
    parsed_rows.append(parsed)

parsed = pd.DataFrame(parsed_rows)
print(parsed[['slug', 'headline', 'date_text', 'n_paragraphs']].to_string(index=False))
print()
print('First lead text from each article:')
for _, r in parsed.iterrows():
    print(f"\\n[{r.slug}]")
    print(f"  {(r.lead_text or '(none)')[:280]}")


## 13. Run the extractor on each parsed article

Same schema. Same system prompt. Same Anthropic call. Each parsed `lead_text` becomes one user message.


In [ ]:
extracted_rows = []
for _, p in parsed.iterrows():
    if not p.lead_text:
        extracted_rows.append({**p.to_dict(), 'extracted_json': None,
                               'validators_passed': False,
                               'validator_errors': 'no parsed lead text'})
        continue
    data, resp = extract_one(p.lead_text)
    ok, errs = validate_record(data) if data is not None else (False, ['JSON parse failed'])
    extracted_rows.append({
        **p.to_dict(),
        'extracted_json': json.dumps(data) if data is not None else None,
        'input_tokens': resp.usage.input_tokens,
        'output_tokens': resp.usage.output_tokens,
        'validators_passed': ok,
        'validator_errors': '; '.join(errs) if errs else '',
    })

extracted = pd.DataFrame(extracted_rows)
print(f'\\nValidator pass rate on end-to-end: {extracted.validators_passed.mean()*100:.0f}%')
print(extracted[['slug', 'validators_passed', 'extracted_json']].to_string(index=False))


## 14. Save the full row, audit log, and cost summary

One CSV per stage, joined on `slug`. The pipeline is restartable from any stage because every stage reads from disk and writes to disk.


In [ ]:
OUT_DIR = Path('/content/drive/MyDrive/day3')
manifest.to_csv(OUT_DIR / 'crawl_manifest.csv', index=False)
parsed.to_csv(OUT_DIR / 'parsed.csv', index=False)
extracted.to_csv(OUT_DIR / 'extracted.csv', index=False)
audit.to_csv(OUT_DIR / 'extraction_audit.csv', index=False)

# One full row inspection
full = manifest.merge(parsed, on='slug').merge(
    extracted[['slug', 'extracted_json', 'validators_passed', 'validator_errors']],
    on='slug',
)
print('\\n=== One full row ===')
for col in ['url', 'status', 'fetched_at', 'raw_path', 'headline', 'date_text',
            'n_paragraphs', 'extracted_json', 'validators_passed']:
    if col in full.columns:
        val = full.iloc[0][col]
        print(f'  {col:>22}: {str(val)[:240]}')

# Cost summary
HAIKU_INPUT_PER_MTOK  = 1.00   # USD per 1M input tokens (verify before workshop)
HAIKU_OUTPUT_PER_MTOK = 5.00
HAIKU_CACHE_PER_MTOK  = 0.10
in_tok    = audit.input_tokens.sum() + extracted.input_tokens.fillna(0).sum()
cache_tok = audit.cache_read_tokens.sum()
out_tok   = audit.output_tokens.sum() + extracted.output_tokens.fillna(0).sum()
cost = (in_tok / 1e6 * HAIKU_INPUT_PER_MTOK
        + cache_tok / 1e6 * HAIKU_CACHE_PER_MTOK
        + out_tok / 1e6 * HAIKU_OUTPUT_PER_MTOK)
print(f'\\n=== Cost summary ===')
print(f'  input tokens:        {int(in_tok):>9}')
print(f'  cache-read tokens:   {int(cache_tok):>9}')
print(f'  output tokens:       {int(out_tok):>9}')
print(f'  estimated cost:      ${cost:.4f}  USD')
print(f'  (verify Anthropic pricing 1-2 weeks before workshop day)')


## 15. No-key fallback. Qwen 2.5-1.5B on free Colab

If you would rather not add a credit card to Anthropic, this cell shows the same workflow with an open-weights model running on free Colab GPU. Output quality is lower than Haiku 4.5 and is not suitable for a published paper. Useful for learning the workflow.

Skip this cell if you ran the Anthropic version above.


In [ ]:
# Uncomment to run.
#
# !pip install -q torch transformers accelerate
#
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# QWEN_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
# tok = AutoTokenizer.from_pretrained(QWEN_ID)
# qmodel = AutoModelForCausalLM.from_pretrained(QWEN_ID, torch_dtype=torch.float16, device_map='auto')
#
# def qwen_extract(lead_text):
#     msgs = [
#         {'role': 'system', 'content': SYSTEM_PROMPT},
#         {'role': 'user',   'content': lead_text},
#     ]
#     inp = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(qmodel.device)
#     out = qmodel.generate(inp, max_new_tokens=200, do_sample=False, pad_token_id=tok.eos_token_id)
#     text = tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
#     try:
#         return json.loads(text)
#     except json.JSONDecodeError:
#         return None
#
# qwen_extract(leads.iloc[0]['text'])
print('Qwen fallback cell. See comments above to enable.')


## What we just did

1. Defined a four-field schema with closed taxonomies and pinned it in the system prompt.
2. Ran fifty extraction calls with prompt caching. Reported the schema-validation pass rate.
3. Re-ran a ten-lead subset twice at temperature zero and reported the determinism rate.
4. Audited five random outputs by hand. The smallest defensible spot-check budget.
5. Read CSIS robots.txt on screen and saved a copy.
6. Built a polite fetcher with a real User-Agent, timeout, and status-code check.
7. Crawled five articles with one-second pauses, stored raw HTML, parsed with BeautifulSoup.
8. Ran the extractor on each parsed lead. Validated. Wrote the audit log and cost summary.
9. Saved one CSV per stage so the pipeline restarts from any stage.

**For the paper.** Slide 28 of the deck is the appendix-checklist version of this notebook. The gold set is the missing piece. When you have one for your concept, the validators in section 5 become field-level F1 against truth. Everything else stays.

**Thanks for spending three days here.** The replication package, including all three demo notebooks, lives at github.com/jacqpark/instats-llm-workshop.
